In [1]:
from pyprims.notebook import Session, Dashboard, figure, plot_batch, read_batch
from IPython.display import Image

def show_model(path):
    """Print a model file."""
    print(open(path, encoding="utf-8").read())

In [2]:
s = Session(seed=1)

In [6]:
%%prims s
// First example model: counting
// Adapted from the ACT-R tutorial by Niels Taatgen
//

define task count {
  initial-skills: (count) // In this example the task and the goal are the same
  default-activation: 1.0 // All chunks defined in this model receive a fixed baselevel activation of 1.0
  ol: t
  rt: -2.0
  lf: 0.2
  default-operator-self-assoc: 0.0
  egs: 0.05
  retrieval-reinforces: t
}


define skill count {
//  Start counting
  operator start-count {
   V1 <> nil  // There has to be an input with the starting number
   WM1 = nil  // Working Memory should be empty
  ==>
   V1 -> WM1 // Copy the start number to working memory
   count-fact -> RT1 // Start retrieving the next number
   V1 -> RT2
   say -> AC1 // Say the current number
   V1 -> AC2
   }

//    Iterate as long as count isn't done
  operator iterate {
    RT2 = WM1
    V2 <> WM1
   ==>
    RT3 -> WM1
    count-fact -> RT1
    RT3 -> RT2
    say -> AC1
    RT3 ->AC2
  }

//    Stop when reaching final number
  operator final {
    V2 = WM1
   ==>
    say -> AC1
    stop -> AC2
    nil -> G1
  }

}

define facts {
    (cf1 count-fact  one  two)
    (cf2 count-fact  two  three)
    (cf3 count-fact  three  four)
    (cf4 count-fact  four  five)
    (cf5 count-fact  five  six)
    (cf6 count-fact six seven)
    (cf7 count-fact seven eight)
    (cf8 count-fact eight nine)
    (cg9 count-fact nine ten)
}

define action say {
  latency: 0.3
  noise: 0.1
  distribution: uniform
  output: Saying
}

define script {
	set-graph-title("Reaction time")
    digits = ["one","two","three","four","five","six","seven",
        "eight","nine","ten"]
    start = random(3)
    end = start + 1 + random(3)
    print("Counting from",digits[start],"to",digits[end])
    screen(digits[start],digits[end])
    run-until-action("say","stop")
    issue-reward()
    trial-end()
}

Loaded task 'count' into s (tasks: count)


In [8]:
s.chunk("start-count")

operator start-count {
    V1<>nil
    WM1=nil
==>
    V1->WM1
    count-fact->RT1
    V1->RT2
    say->AC1
    V1->AC2
}

Total activation = 2.766
Baselevel activation = 2.456  (fixed 1.000, refs 1)
Current noise = 0.309
Total spreading activation = 0.000

Spreading activation calculation

Associations (Sji) to this chunk
  count                                      3.000  (updates 0)
  final                                      1.000  (updates 0)
  iterate                                    1.000  (updates 0)
  start-count                                0.000  (updates 0)

In [14]:
%%prims add --reset
// Add numbers by counting
// The script assumes that the final action is "answer"

define task add {
    initial-skills: (add)
    default-activation: 1.0
}


define skill add {
    // Write your four operators here
// 1️⃣ INIT: put V1 in WM1, zero in WM2, ask for count fact about V1
  operator start-add {
    V1 <> nil          // there's a first addend
    WM1 = nil          // memory is empty (not started yet)
   ==>
    V1 -> WM1          // WM1 = first addend (current sum)
    zero -> WM2        // WM2 = current count (starts at zero)
    count-fact -> RT1  // retrieve: what comes after V1?
    V1 -> RT2
    // say -> AC1 // Say the current number, not needed
    V1 -> AC2
  }

// 2️⃣ HARVEST SUM: get next sum from WM1, ask for count fact about WM2
  operator update-sum {
    RT2 = WM1          // retrieved next-number matches current sum
    WM2 <> V2 //not correct answer
   ==>
    RT3 -> WM1         // WM1 = next sum (climb sum ladder) by one
    count-fact -> RT1  // retrieve: what comes after WM2 (the count)?
    WM2 -> RT2 // ask about the current count

  }
    // 3️⃣ HARVEST COUNT: get next count from WM2, check not done, ask for count fact about WM1
  operator update-count {
    RT2 = WM2          // retrieved next-count matches current count
    WM2 <> V2          // haven't reached the second addend yet
   ==>
    RT3 -> WM2         // WM2 = next count (climb count ladder)
    count-fact -> RT1  // retrieve: what comes after WM1 (the sum)?
    WM1 -> RT2
  }
    // 4️⃣ ANSWER: count reached V2 — say the sum in WM1
  operator give-answer {
    WM2 = V2           // count = second addend, we're done!
   ==>
    answer -> AC1      // the action the script waits for
    WM1 -> AC2         // say the sum
    nil -> G1          // clean up goal
  }
}

define facts {
    (cf0 count-fact  zero  one :activation 3.0)
    (cf1 count-fact  one  two :activation 3.0)
    (cf2 count-fact  two  three :activation 3.0)
    (cf3  count-fact  three  four :activation 3.0)
    (cf4  count-fact  four  five :activation 3.0)
    (cf5  count-fact  five  six :activation 3.0)
    (cf6  count-fact  six  seven :activation 3.0)
    (cf7  count-fact  seven  eight :activation 3.0)
    (cf8  count-fact  eight  nine :activation 3.0)
    (cf9  count-fact  nine  ten :activation 3.0)
}

define action say {
  latency: 0.3
  noise: 0.1
  distribution: uniform
  output: Saying
}

define script {
    digits = ["zero","one","two","three","four","five","six","seven","eight","nine","ten"]
    num1 = random(4) + 1
    num2 = random(4) + 1
    print("Adding",digits[num1],"and",digits[num2])
    screen(digits[num1],digits[num2])
    run-until-action("answer")
    issue-reward()
    trial-end()
}

Loaded task 'add' into add (tasks: add)


In [13]:
add.run(1)
print(add.trace(2))

Adding two and three 
6.08  Starting new trial
6.08  *** Retrieved operator start-add with latency 0.003
7.93  Retrieving cf2 (latency = 0.008)
7.94  *** Retrieved operator update-sum with latency 0.002
9.19  Retrieving cf0 (latency = 0.015)
9.21  *** Retrieved operator update-count with latency 0.001
10.46  Retrieving cf3 (latency = 0.009)
10.82  *** Retrieved operator update-sum with latency 0.001
12.07  Retrieving cf1 (latency = 0.010)
12.43  *** Retrieved operator update-count with latency 0.001
13.68  Retrieving cf4 (latency = 0.017)
14.05  *** Retrieved operator update-sum with latency 0.001
15.30  Retrieving cf2 (latency = 0.021)
15.67  *** Retrieved operator update-count with latency 0.001
16.92  Retrieving cf5 (latency = 0.012)
16.94  *** Retrieved operator give-answer with latency 0.001
17.89  answer-ing five-
17.94  End of trial


In [16]:
Dashboard(add)